In [ ]:
# 실습 준비 — 05주차 이산형 확률변수
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = []

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 이산형 확률분포

## 1차원 이산형 확률분포

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%precision 3
%matplotlib inline

### 1차원 이산확률분포의 정의

#### 이론 요약
- 이산형 확률변수 $X$의 가능한 값 집합 $X = \{x_1, \dots, x_k\}$  
- 확률질량함수(PMF): <tr>
$f(x) = P(X = x_i)$
- $\sum_i p(x_i) = 1$

In [ ]:
# 1) 확률변수 X의 가능한 값 정의
x_set = np.array([1, 2, 3, 4, 5, 6])

In [ ]:
# 2) 확률함수 정의
def f(x):
    if x in x_set:
        return x / 21
    else:
        return 0

In [ ]:
X=[x_set, f]

In [ ]:
# 확률 p_k를 구한다
prob = np.array([f(x_k) for x_k in x_set])

In [ ]:
# 3) 분포표 작성
df = pd.DataFrame({'x': x_set, 'f(x)': prob})
df

In [ ]:
# 4) 확률이 0이상인지 검증
np.all(prob >= 0)

In [ ]:
# 5) 확률 합이 1인지 검증
np.sum(prob)

In [ ]:
# 6) 막대그래프작성
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
ax.bar(x_set, prob)
ax.set_xlabel('value')
ax.set_ylabel('probability')

plt.show()

#### 공정한 주사위 확률분포 실습
1) 확률변수 X의 가능한 값 정의 (1~6)
2) 각 값의 확률 벡터 P 생성 (공정한 주사위 → 모두 1/6)
3) 확률분포표(DataFrame) 구성
4) 확률이 0이상인지 검증
5) 확률 합이 1인지 검증
6) [선택] 막대그래프 작성

In [ ]:
# 여기에 코드를 작성하세요.

### 누적분포함수 (Dumulative Distribution Function, CDF)
- 확률변수 $X$가 $x$이하가 될 때의 확률
- $F(x) = P(X <= x) = \sum_{x_k >= x}f(x_k)$

In [ ]:
# cdf 정의
def F(x):
    return np.sum([f(x_k) for x_k in x_set if x_k <= x])

In [ ]:
# 주사위눈이 3이하일 확률
F(3)

In [ ]:
# 주사위눈이 6이하일 확률
# 6 은 확률변수가 가질 수 있는 가장 큰 값이므로 누적확률은 1 이 된다.


In [ ]:
# 확률변수 Y = 2X + 3
y_set = np.array([2 * x_k + 3 for x_k in x_set])
# 확률
prob = np.array([f(x_k) for x_k in x_set])

In [ ]:
# 분포표 작성
df = pd.DataFrame({'x': x_set, 'f(x)': prob, 'y': y_set})
df

### 1차원 이산형 확률변수의 지표

#### 평균 (기댓값: Expected value)
- 무제한 실행한 실현값의 평균 (불가능)
- 이산형: 확률변수가 취할 수 있는 값과 그 확률의 곱의 총합
- $E(X) = \sum_k x_k f(x_k)$

In [ ]:
np.sum([x_k * f(x_k) for x_k in x_set])

In [ ]:
sample = np.random.choice(x_set, int(1e6), p=prob)
np.mean(sample)

In [ ]:
def E(X, g=lambda x: x):
    x_set, f = X
    return np.sum([g(x_k) * f(x_k) for x_k in x_set])

In [ ]:
E(X)

#### 기댓값의 선형성
$E(aX + b) = a E(X) + b$

In [ ]:
E(X, g=lambda x: 2*x + 3)

In [ ]:
2 * E(X) + 3

#### 분산
- 편차제곱의 기댓값
- $V(X) = \sum_k (x_k - \mu)^2 f(x_k)$
- 변환함수 $g(X)$에 대한 일반화: $V(g(X)) = \sum_k (g(x_k) - E(g(X)))^2 f(x_k)$

In [ ]:
mean = E(X)
np.sum([(x_k-mean)**2 * f(x_k) for x_k in x_set])

In [ ]:
sample = np.random.choice(x_set, int(1e6), p=prob)
np.var(sample)

In [ ]:
# 인수가 g인 확률변수
def V(X, g=lambda x: x):
    x_set, f = X
    mean = E(X, g)
    return np.sum([(g(x_k)-mean)**2 * f(x_k) for x_k in x_set])

In [ ]:
# g를 지정 안함
V(X)

In [ ]:
# g를 지정함
V(X, lambda x: 2*x + 3)

#### 분산의 공식
- $V(aX + b) = a^2 V(X)$

In [ ]:
2**2 * V(X)

## 2차원 이산형 확률분포

### 2차원 이산형 확률분포의 정의
- 확률변수 $(X,Y)$의 움직임을 동시에 고려한 분포
- 예시) X: 주사위 A + B, Y: A

In [ ]:
x_set = np.arange(2, 13)
y_set = np.arange(1, 7)

In [ ]:
def f_XY(x, y):
    if 1 <= y <=6 and 1 <= x - y <= 6:
        return y * (x-y) / 441
    else:
        return 0

In [ ]:
XY = [x_set, y_set, f_XY]

In [ ]:
prob = np.array([[f_XY(x_i, y_j) for y_j in y_set]
                 for x_i in x_set])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)

c = ax.pcolor(prob)
ax.set_xticks(np.arange(prob.shape[1]) + 0.5, minor=False)
ax.set_yticks(np.arange(prob.shape[0]) + 0.5, minor=False)
ax.set_xticklabels(np.arange(1, 7), minor=False)
ax.set_yticklabels(np.arange(2, 13), minor=False)
# y축을 내림차순의 숫자가 되게 하여, 위 아래를 역전시킨다
ax.invert_yaxis()
# x축의 눈금을 그래프 위쪽에 표시
ax.xaxis.tick_top()
fig.colorbar(c, ax=ax)
plt.show()

In [ ]:
np.all(prob >= 0)

In [ ]:
np.sum(prob)

#### 주변확률분포
- 확률 X의 확률 함수를 알고싶은경우
- 확률 Y의 영향력을 제거
- $f_XY$에서 $y$가 취할 수 있는 값을 모두 대입한 뒤 합함
- $f_X(x) = \sum_k f_{XY}(x,y)$

In [ ]:
def f_X(x):
    return np.sum([f_XY(x, y_k) for y_k in y_set])

In [ ]:
def f_Y(y):
    return np.sum([f_XY(x_k, y) for x_k in x_set])

In [ ]:
X = [x_set, f_X]
Y = [y_set, f_Y]

In [ ]:
prob_x = np.array([f_X(x_k) for x_k in x_set])
prob_y = np.array([f_Y(y_k) for y_k in y_set])

fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

ax1.bar(x_set, prob_x)
ax1.set_title('X_marginal probability distribution')
ax1.set_xlabel('X_value')
ax1.set_ylabel('probability')
ax1.set_xticks(x_set)

ax2.bar(y_set, prob_y)
ax2.set_title('Y_marginal probability distribution')
ax2.set_xlabel('Y_value')
ax2.set_ylabel('probability')

plt.show()

### 2차원 이산형 확률분포의 지표

#### 기댓값
- $\mu_X = E(X) = \sum_i \sum_j x_i f_{XY} (x_i, y_j)$

####$g(X,Y)$의 기댓값
- $E(g(X,Y)) = \sum_i \sum_j g(x_i,y_i) f_{XY} (x_i, y_j)$

In [ ]:
np.sum([x_i * f_XY(x_i, y_j) for x_i in x_set for y_j in y_set])

In [ ]:
def E(XY, g):
    x_set, y_set, f_XY = XY
    return np.sum([g(x_i, y_j) * f_XY(x_i, y_j)
                   for x_i in x_set for y_j in y_set])

In [ ]:
mean_X = E(XY, lambda x, y: x)
mean_X

In [ ]:
mean_Y = E(XY, lambda x, y: y)
mean_Y

#### 기댓값의 선형성
- $E(aX + bY) = aE(X) + bE(Y)

In [ ]:
a, b = 2, 3

In [ ]:
E(XY, lambda x, y: a*x + b*y)

In [ ]:
a * mean_X + b * mean_Y

#### 분산
- X의 편차 제곱의 기댓값
- $\sigma^2_X = V(X) = \sum_i \sum_j (x_i - \mu_X)^2 f_{XY}(x_i, y_j)$

In [ ]:
np.sum([(x_i-mean_X)**2 * f_XY(x_i, y_j)
       for x_i in x_set for y_j in y_set])

#### $g(X,Y)$의 분산

- $V(g(X, Y)) = \sum_i \sum_j (x_i - \mu_x)^2 f_{XY}(x_i, y_j)$

In [ ]:
def V(XY, g):
    x_set, y_set, f_XY = XY
    mean = E(XY, g)
    return np.sum([(g(x_i, y_j)-mean)**2 * f_XY(x_i, y_j)
                   for x_i in x_set for y_j in y_set])

In [ ]:
var_X = V(XY, g=lambda x, y: x)
var_X

In [ ]:
var_Y = V(XY, g=lambda x, y: y)
var_Y

In [ ]:
def Cov(XY):
    x_set, y_set, f_XY = XY
    mean_X = E(XY, lambda x, y: x)
    mean_Y = E(XY, lambda x, y: y)
    return np.sum([(x_i-mean_X) * (y_j-mean_Y) * f_XY(x_i, y_j)
                    for x_i in x_set for y_j in y_set])

In [ ]:
cov_xy = Cov(XY)
cov_xy

In [ ]:
V(XY, lambda x, y: a*x + b*y)

In [ ]:
a**2 * var_X + b**2 * var_Y + 2*a*b * cov_xy

In [ ]:
cov_xy / np.sqrt(var_X * var_Y)